##### Define notebook Parameters

Pass through th pl_worker

In [26]:
# Framework-compatible parameters with manual fallbacks
import json
import uuid

# Basic notebook runtime values
task_id = 50
task_name = "Load taxonomy to silver"
lineage_id = str(uuid.uuid4())
limit_rows_for_debugging = False
prev_wm = None
curr_wm = None

# Source settings > what data to read
source_settings = json.dumps(
    {
        "source_name": "taxonomy",
        "source_type": "shortcut",
        "source_location": "external_curated_taxonomy",
        "source_schema": "external_curated_taxonomy",
        "table_config": [
            {
                "source_table": "curated_band_level",
                "target_table": "stage_taxonomy_curated_band_level",
                "target_pk": ["band_level_code"],
                "is_incremental": 0,
                "incremental_column": ""
            },
            {
                "source_table": "curated_bill_type",
                "target_table": "stage_taxonomy_curated_bill_type",
                "target_pk": ["bill_type_code"]
            },
            {
                "source_table": "curated_client_sector",
                "target_table": "stage_taxonomy_curated_client_sector",
                "target_pk": ["sic_code"]
            },
            {
                "source_table": "curated_company",
                "target_table": "stage_taxonomy_curated_company",
                "target_pk": ["company_code"]
            },
            {
                "source_table": "curated_country",
                "target_table": "stage_taxonomy_curated_country",
                "target_pk": ["country_code_iso2"]
            },
            {
                "source_table": "curated_currency",
                "target_table": "stage_taxonomy_curated_currency",
                "target_pk": ["currency_code"]
            },
            {
                "source_table": "curated_employee_sub_group",
                "target_table": "stage_taxonomy_curated_employee_sub_group",
                "target_pk": ["employee_sub_group_code"]
            },
            {
                "source_table": "curated_entry_classification",
                "target_table": "stage_taxonomy_curated_entry_classification",
                "target_pk": ["entry_classification_code"]
            },
            {
                "source_table": "curated_matter_billing_frequency",
                "target_table": "stage_taxonomy_curated_matter_billing_frequency",
                "target_pk": ["matter_billing_frequency_code"]
            },
            {
                "source_table": "curated_matter_billing_method",
                "target_table": "stage_taxonomy_curated_matter_billing_method",
                "target_pk": ["matter_billing_method_code"]
            },
            {
                "source_table": "curated_matter_category",
                "target_table": "stage_taxonomy_curated_matter_category",
                "target_pk": ["matter_category_code"]
            },
            {
                "source_table": "curated_matter_entry_type",
                "target_table": "stage_taxonomy_curated_matter_entry_type",
                "target_pk": ["matter_entry_type_code"]
            },
            {
                "source_table": "curated_matter_sector",
                "target_table": "stage_taxonomy_curated_matter_sector",
                "target_pk": ["sic_code"]
            },
            {
                "source_table": "curated_matter_status",
                "target_table": "stage_taxonomy_curated_matter_status",
                "target_pk": ["matter_status_code"]
            },
            {
                "source_table": "curated_nature_of_proceeding",
                "target_table": "stage_taxonomy_curated_nature_of_proceeding",
                "target_pk": ["nature_of_proceeding_code"]
            },
            {
                "source_table": "curated_office",
                "target_table": "stage_taxonomy_curated_office",
                "target_pk": ["office_code"]
            },
            {
                "source_table": "curated_partner_function",
                "target_table": "stage_taxonomy_curated_partner_function",
                "target_pk": ["partner_function_code"]
            },
            {
                "source_table": "curated_practice_group",
                "target_table": "stage_taxonomy_curated_practice_group",
                "target_pk": ["practice_group_code"]
            },
            {
                "source_table": "curated_team",
                "target_table": "stage_taxonomy_curated_team",
                "target_pk": ["team_code", "member_firm_code"]
            },
            {
                "source_table": "curated_transaction_document_type",
                "target_table": "stage_taxonomy_curated_transaction_document_type",
                "target_pk": ["transaction_document_type_code"]
            },
            {
                "source_table": "curated_work_type",
                "target_table": "stage_taxonomy_curated_work_type",
                "target_pk": ["technical_area_code"]
            }
        ]
    }
)

# Target settings > here/how to write
target_settings = json.dumps(
    {
        "target_lakehouse": "lh_canada_global_mds",
        "target_schema": "stage_taxonomy",
        "target_location": "Files/stage/taxonomy",
        "target_load_strategy": "overwrite"
    }
)

# Connection settings > how to connect/authenticate
source_connection_settings  = "{}"

StatementMeta(, e844e3b5-9d1c-4c80-840c-6cc4dd59e35a, 43, Finished, Available, Finished, False)

##### Define functions and logging

Used for import functions

In [27]:
%run nb_transformation_shared_functions

StatementMeta(, e844e3b5-9d1c-4c80-840c-6cc4dd59e35a, 49, Finished, Available, Finished, True)

In [28]:
# Set up standard framework logging
setup_logging()
logger = logging.getLogger("LoadTaxonomyToStage")
logger.setLevel(logging.INFO)

StatementMeta(, e844e3b5-9d1c-4c80-840c-6cc4dd59e35a, 50, Finished, Available, Finished, False)

##### Define variables or parameters

Can manually run or be through worker

In [29]:
# Accept either JSON strings from pipeline or already-parsed dicts for manual runs.
source_settings = json.loads(source_settings) if isinstance(source_settings, str) else source_settings
target_settings = json.loads(target_settings) if isinstance(target_settings, str) else target_settings
source_connection_settings = json.loads(source_connection_settings) if isinstance(source_connection_settings, str) else source_connection_settings

# Source configuration
source_name = source_settings.get("source_name")
source_type = source_settings.get("source_type")
source_location = source_settings.get("source_location")
source_schema = source_settings.get("source_schema")
table_config = source_settings.get("table_config", [])

# Target configuration
target_lakehouse = target_settings.get("target_lakehouse")
target_schema = target_settings.get("target_schema")
target_location = target_settings.get("target_location")
target_load_strategy = target_settings.get("target_load_strategy")

# Source connection configuration
server_name = (
    source_connection_settings.get("server_name")
    or source_connection_settings.get("jdbc_hostname")
    or source_connection_settings.get("server")
)

# Debug output
if limit_rows_for_debugging:
    print(f"source_name: {source_name}")
    print(f"source_type: {source_type}")
    print(f"source_location: {source_location}")
    print(f"source_schema: {source_schema}")
    print(f"target_schema: {target_schema}")
    print(f"target_location: {target_location}")
    print(f"target_load_strategy: {target_load_strategy}")
    display(table_config)

StatementMeta(, e844e3b5-9d1c-4c80-840c-6cc4dd59e35a, 51, Finished, Available, Finished, False)

##### Build source connection

Creates connection to get source data

In [30]:
# Build source connection if this source needs one.
jdbc_url, connection_properties = build_source_connection(
    source_type=source_type,
    source_location=source_location,
    source_connection_settings=source_connection_settings
)

# Debug output
if limit_rows_for_debugging:
    print(f"jdbc_url: {jdbc_url}")

StatementMeta(, e844e3b5-9d1c-4c80-840c-6cc4dd59e35a, 52, Finished, Available, Finished, False)

##### Ingest source table

Read and dedup clean source

In [31]:
# Read source tables and create src_* temp views.
source_dfs = {}

for cfg in table_config:
    # Get the source table name from the table config.
    source_table = cfg.get("source_table")

    # Create a standard view name and read the source table.
    source_view = f"src_{to_snake_case(source_table)}"
    source_full_name = f"{source_schema}.{source_table}"
    
    # Read data from sql/shortcut/lakehouse
    df_source = read_source_table(
        source_full_name=source_full_name,
        jdbc_url=jdbc_url,
        connection_properties=connection_properties
    )

    # Clean, register, and save the source table for later steps.
    df_source = remove_exact_duplicates(df_source)
    df_source.createOrReplaceTempView(source_view)
    source_dfs[source_table] = df_source

    if limit_rows_for_debugging:
        print(f"Source preview for {source_view} ({source_full_name})")
        display(df_source.limit(3))

StatementMeta(, e844e3b5-9d1c-4c80-840c-6cc4dd59e35a, 53, Finished, Available, Finished, False)

##### Transform source table

Applies mapping and transformation as needed

In [32]:
# Optional transformation step.
mapped_dfs = {}

StatementMeta(, e844e3b5-9d1c-4c80-840c-6cc4dd59e35a, 54, Finished, Available, Finished, False)

##### Add metadata to source table

Applies metadata and _hashed_pk

In [33]:
# Add metadata to final target data.
target_dfs = {}

for cfg in table_config:
    # Get source and target details.
    source_table = cfg.get("source_table")
    target_table = cfg.get("target_table")
    target_pk = cfg.get("target_pk", [])

    # Use transformed data when it exists; otherwise use the ingested source data.
    df_target = mapped_dfs.get(target_table, source_dfs[source_table])

    # Add hashes only for merge loads.
    if target_load_strategy == "merge-scd1":
        df_target = add_hash_columns(df_target, target_pk)

    # Add standard framework metadata columns.
    df_target = add_metadata_v2(
        df=df_target,
        source_system=source_name,
        source_table=f"{source_schema}.{source_table}",
        target_table=target_table,
        lineage_id=lineage_id
    )

    # Save final DataFrame for validation and load.
    target_dfs[target_table] = df_target

    if limit_rows_for_debugging:
        print(f"Target preview for {target_table}")
        display(df_target.limit(3))

StatementMeta(, e844e3b5-9d1c-4c80-840c-6cc4dd59e35a, 55, Finished, Available, Finished, False)

##### Check duplicates

Return duplicates before load_to_target 

In [34]:
# Check duplicate business keys before load
for cfg in table_config:
    # Get table definition from source_settings
    target_table = cfg.get("target_table")
    target_pk = cfg.get("target_pk")
    
    # Call nb_transformation_shared_functions
    validate_duplicates(target_dfs[target_table], target_pk)

StatementMeta(, e844e3b5-9d1c-4c80-840c-6cc4dd59e35a, 56, Finished, Available, Finished, False)

Total duplicate ['band_level_code'] groups: 0
Total duplicate ['bill_type_code'] groups: 0
Total duplicate ['sic_code'] groups: 0
Total duplicate ['company_code'] groups: 0
Total duplicate ['country_code_iso2'] groups: 0
Total duplicate ['currency_code'] groups: 0
Total duplicate ['employee_sub_group_code'] groups: 0
Total duplicate ['entry_classification_code'] groups: 0
Total duplicate ['matter_billing_frequency_code'] groups: 0
Total duplicate ['matter_billing_method_code'] groups: 0
Total duplicate ['matter_category_code'] groups: 0
Total duplicate ['matter_entry_type_code'] groups: 0
Total duplicate ['sic_code'] groups: 0
Total duplicate ['matter_status_code'] groups: 0
Total duplicate ['nature_of_proceeding_code'] groups: 0
Total duplicate ['office_code'] groups: 0
Total duplicate ['partner_function_code'] groups: 0
Total duplicate ['practice_group_code'] groups: 0
Total duplicate ['team_code', 'member_firm_code'] groups: 0
Total duplicate ['transaction_document_type_code'] group

##### Load to target

Runs load to target based on load_strategy

In [35]:
# Load final DataFrames to silver taxonomy tables
load_results = []

for cfg in table_config:
    # Get table definition from source_settings
    target_table = cfg.get("target_table")

    # Set target table and location
    target_full_name = f"{target_schema}.{target_table}"
    target_location_path = f"{target_location}/{target_table}"
    
    # Lookup target table on target_dfs
    df_target = target_dfs[target_table]

    # Call nb_transformation_shared_functions
    result = load_to_target(df_target, target_full_name, target_load_strategy, target_location_path)

    # Save metrics for pipeline
    load_results.append({
        "target_table": target_full_name,
        "rows_inserted": result["rows_inserted"],
        "rows_updated": result["rows_updated"],
        "rows_read": result["rows_read"],
        "rows_copied": result["rows_copied"]
    })

display(load_results)

StatementMeta(, e844e3b5-9d1c-4c80-840c-6cc4dd59e35a, 57, Finished, Available, Finished, False)

2026-07-03 16:11:07,267 UTC - INFO - Overwriting records in stage_taxonomy.stage_taxonomy_curated_band_level (LoadTaxonomyToStage)
2026-07-03 16:11:13,275 UTC - INFO - Overwriting records in stage_taxonomy.stage_taxonomy_curated_bill_type (LoadTaxonomyToStage)
2026-07-03 16:11:18,460 UTC - INFO - Overwriting records in stage_taxonomy.stage_taxonomy_curated_client_sector (LoadTaxonomyToStage)
2026-07-03 16:11:25,071 UTC - INFO - Overwriting records in stage_taxonomy.stage_taxonomy_curated_company (LoadTaxonomyToStage)
2026-07-03 16:11:30,285 UTC - INFO - Overwriting records in stage_taxonomy.stage_taxonomy_curated_country (LoadTaxonomyToStage)
2026-07-03 16:11:35,161 UTC - INFO - Overwriting records in stage_taxonomy.stage_taxonomy_curated_currency (LoadTaxonomyToStage)
2026-07-03 16:11:40,141 UTC - INFO - Overwriting records in stage_taxonomy.stage_taxonomy_curated_employee_sub_group (LoadTaxonomyToStage)
2026-07-03 16:11:44,956 UTC - INFO - Overwriting records in stage_taxonomy.stage_

SynapseWidget(Synapse.DataFrame, affb9bac-b4b0-41af-abc6-9a1cf00a2b85)